# TabPFN-3 + RealMLP + XGBoost Stacker
Stacks 3 RealMLP + 3 XGBoost models using TabPFN-3 as meta-learner with raw features.

In [ ]:
!uv pip install tabpfn --system -q
import os
os.makedirs("/tmp/tabpfn_cache", exist_ok=True)
os.environ["TABPFN_MODEL_CACHE_DIR"] = "/tmp/tabpfn_cache"
# Also try the Kaggle model path if available
kaggle_model = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
if os.path.exists(kaggle_model):
    os.environ["TABPFN_MODEL_CACHE_DIR"] = kaggle_model
    print("Using Kaggle model cache:", kaggle_model)
else:
    print("Kaggle model not found, using /tmp/tabpfn_cache (will download)")


In [ ]:
import numpy as np, pandas as pd, warnings, gc
from sklearn.metrics import balanced_accuracy_score, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import torch
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv', index_col='id')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv', index_col='id')
y = train['class'].map({'GALAXY':0,'QSO':1,'STAR':2}).values
target_map = {'GALAXY':0,'QSO':1,'STAR':2}
inv_map = {v:k for k,v in target_map.items()}
print(f'Train: {len(train):,} | Test: {len(test):,}')

In [ ]:
import numpy as np, pandas as pd, warnings, gc, os
from sklearn.metrics import balanced_accuracy_score, accuracy_score
from sklearn.model_selection import StratifiedKFold
import torch
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/train.csv", index_col="id")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/test.csv", index_col="id")
y = train["class"].map({"GALAXY":0,"QSO":1,"STAR":2}).values
N = len(train); M = len(test)
print("Train:", N, "| Test:", M)

def prob_to_logit(p):
    p = np.clip(p, 1e-15, 1-1e-15).astype(np.float64)
    return np.clip(np.log(p/(1-p)), -30, 30).astype(np.float32)

DATA = "/kaggle/input/s6e6-stacking-preds"
all_oof = []; all_test = []; names = []

for tag in ["s42", "s123", "s777", "wide", "deep"]:
    oof_p = DATA + "/realmlp_oof_" + tag + ".csv"
    test_p = DATA + "/realmlp_test_" + tag + ".csv"
    if os.path.exists(oof_p):
        o = prob_to_logit(pd.read_csv(oof_p)[["GALAXY","QSO","STAR"]].values[:N])
        t = prob_to_logit(pd.read_csv(test_p)[["GALAXY","QSO","STAR"]].values[:M])
        all_oof.append(o); all_test.append(t); names.append("rm_" + tag)
        print("Loaded rm_" + tag)

for seed in [42, 123, 777]:
    oof_p = DATA + "/xgb_oof_" + str(seed) + ".npy"
    test_p = DATA + "/xgb_test_" + str(seed) + ".npy"
    if os.path.exists(oof_p):
        o = prob_to_logit(np.load(oof_p))
        t = prob_to_logit(np.load(test_p))
        all_oof.append(o); all_test.append(t); names.append("xgb_" + str(seed))
        print("Loaded xgb_" + str(seed))

# Raw features: drop string columns
num_cols = train.drop("class", axis=1).select_dtypes(include=[np.number]).columns.tolist()
raw_train = train[num_cols].values.astype(np.float32)
raw_test = test[num_cols].values.astype(np.float32)
print("Raw numeric features:", len(num_cols))

X_meta_oof = np.hstack(all_oof + [raw_train])
X_meta_test = np.hstack(all_test + [raw_test])
print("Meta features:", X_meta_oof.shape[1], "from", len(names), "models +", len(num_cols), "raw")


In [ ]:
from tabpfn import TabPFNClassifier

print("Training TabPFN-3 stacker...")
clf = TabPFNClassifier(device="cuda", n_estimators=2, balance_probabilities=True)
clf.fit(X_meta_oof, train["class"].values)
print("TabPFN-3 fitted!")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros((N, 3))
for fold, (tr, vl) in enumerate(skf.split(X_meta_oof, y), 1):
    fc = TabPFNClassifier(device="cuda", n_estimators=2, balance_probabilities=True)
    fc.fit(X_meta_oof[tr], train["class"].values[tr])
    oof_preds[vl] = fc.predict_proba(X_meta_oof[vl])
    ba = balanced_accuracy_score(y[vl], np.argmax(oof_preds[vl], 1))
    print("Fold", fold, "BA:", round(ba, 5))
    gc.collect()

overall_ba = balanced_accuracy_score(y, np.argmax(oof_preds, 1))
overall_acc = accuracy_score(y, np.argmax(oof_preds, 1))
print("OOF BA:", round(overall_ba, 5), "Acc:", round(overall_acc, 5))


In [ ]:
test_preds = clf.predict_proba(X_meta_test)
inv_map = {0:"GALAXY", 1:"QSO", 2:"STAR"}
sub = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
sub["class"] = [inv_map[p] for p in np.argmax(test_preds, 1)]
sub.to_csv("submission.csv", index=False)
print("Saved submission.csv")
sub.head()
